# 07 — Factor Research
Evaluate each alpha factor individually using:
- Monthly Spearman IC vs. `target_1m`
- Long-short quintile spread

Use these results to understand which new feature groups add the most signal and 
to inform future feature selection / combination.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

PROCESSED_DIR = "../data/processed/"

## 1. Load data

In [ ]:
monthly = pd.read_csv(
    PROCESSED_DIR + "model_data.csv",
    parse_dates=["Date"]
)

with open(PROCESSED_DIR + "feature_cols.json") as f:
    feature_cols = json.load(f)

monthly = monthly.sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)
target_col = "target_1m"

# Restrict to test period for out-of-sample factor evaluation
test = monthly[monthly["Date"] >= "2021-01-01"].copy()
print("Test set shape:", test.shape)

## 2. Per-factor IC table

In [ ]:
def factor_ic(df, factor, target="target_1m"):
    """Compute mean IC and ICIR for a single factor across all dates."""
    ic_series = (
        df.groupby("Date")
          .apply(
              lambda x: spearmanr(x[factor].dropna(), x.loc[x[factor].notna(), target]).correlation
              if x[factor].notna().sum() > 10 else np.nan,
              include_groups=False
          )
    )
    ic_series = ic_series.dropna()
    mean_ic = ic_series.mean()
    icir    = mean_ic / (ic_series.std() + 1e-8)
    pos_rate = (ic_series > 0).mean()
    return {"mean_ic": mean_ic, "icir": icir, "ic_pos_rate": pos_rate, "n_months": len(ic_series)}

rows = []
for f in feature_cols:
    row = {"factor": f}
    row.update(factor_ic(test, f))
    rows.append(row)

factor_summary = pd.DataFrame(rows).set_index("factor").sort_values("mean_ic", ascending=False)

print("Top 20 factors by Mean IC:")
print(factor_summary.head(20).to_string())

In [ ]:
print("\nBottom 10 factors (weakest/inverse signal):")
print(factor_summary.tail(10).to_string())

## 3. Visualise IC distribution across factors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Mean IC bar chart (top 30)
factor_summary["mean_ic"].head(30).plot(
    kind="barh", ax=axes[0], color="steelblue"
)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Top 30 factors — Mean IC (test set)")
axes[0].invert_yaxis()

# ICIR bar chart (top 30)
factor_summary["icir"].head(30).plot(
    kind="barh", ax=axes[1], color="darkorange"
)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Top 30 factors — ICIR (test set)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 4. Factor group summary
Aggregate by feature group to understand which categories add the most value.

In [ ]:
# Label each factor by its group
GROUP_PATTERNS = {
    "Momentum"        : ["mom_", "ma_"],
    "Volatility"      : ["vol_", "skew_", "max_return_"],
    "Liquidity"       : ["avg_dollar_vol", "rel_volume"],
    "Profitability"   : ["roe", "roa", "margin", "asset_turnover", "equity_ratio",
                         "profit_to_assets", "sales_to_equity"],
    "Valuation"       : ["pb_ratio", "bm_ratio", "pe_ratio", "earnings_yield",
                         "log_market_cap", "dividend_yield", "forecast_earnings_yield",
                         "forecast_dividend_yield"],
    "Growth"          : ["_yoy", "growth_accel"],
    "Forecast"        : ["forecast_sales", "forecast_profit", "forecast_eps",
                         "forecast_roa", "forecast_roe", "forecast_operating",
                         "forecast_ordinary", "revision"],
    "ShareStructure"  : ["treasury", "buyback", "share_count", "shares_outstanding"],
    "FundamentalChange": ["delta_"],
    "Dividend"        : ["dps", "dividend_growth"],
}

def assign_group(factor):
    for group, patterns in GROUP_PATTERNS.items():
        if any(p in factor for p in patterns):
            return group
    return "Other"

factor_summary["group"] = factor_summary.index.map(assign_group)

group_summary = (
    factor_summary
    .groupby("group")
    .agg(mean_ic=("mean_ic", "mean"), icir=("icir", "mean"), n_factors=("mean_ic", "count"))
    .sort_values("mean_ic", ascending=False)
)

print(group_summary.to_string())

## 5. Save factor summary

In [ ]:
factor_summary.to_csv(PROCESSED_DIR + "factor_research_summary.csv")
print("Saved: factor_research_summary.csv")